# Ejercicios ensembling
En este ejercicio vas a realizar prediciones sobre un dataset de ciudadanos indios diabéticos. Se trata de un problema de clasificación en el que intentaremos predecir 1 (diabético) 0 (no diabético).

### 1. Carga las librerias que consideres comunes al notebook

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### 2. Lee los datos de [esta direccion](https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv)
Los nombres de columnas son:
```Python
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
```

In [6]:
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'

In [7]:
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']

In [8]:
df = pd.read_csv(url, sep=",", names=names)

In [9]:
df.head()

,preg,plas,pres,skin,test,mass,pedi,age,class
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   preg    768 non-null    int64  
 1   plas    768 non-null    int64  
 2   pres    768 non-null    int64  
 3   skin    768 non-null    int64  
 4   test    768 non-null    int64  
 5   mass    768 non-null    float64
 6   pedi    768 non-null    float64
 7   age     768 non-null    int64  
 8   class   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [11]:
df.describe()

,preg,plas,pres,skin,test,mass,pedi,age,class
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


### 3. Bagging
Para este apartado tendrás que crear un ensemble utilizando la técnica de bagging ([BaggingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html)), mediante la cual combinarás 100 [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html). Recuerda utilizar también [cross validation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html) con 10 kfolds.

**Para este apartado y siguientes, no hace falta que dividas en train/test**, por hacerlo más sencillo. Simplemente divide tus datos en features y target.

Establece una semilla

In [12]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import KFold, cross_val_score

In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [14]:
X = df.loc[:, 'preg':'age'].values
X

array([[  6.   , 148.   ,  72.   , ...,  33.6  ,   0.627,  50.   ],
       [  1.   ,  85.   ,  66.   , ...,  26.6  ,   0.351,  31.   ],
       [  8.   , 183.   ,  64.   , ...,  23.3  ,   0.672,  32.   ],
       ...,
       [  5.   , 121.   ,  72.   , ...,  26.2  ,   0.245,  30.   ],
       [  1.   , 126.   ,  60.   , ...,  30.1  ,   0.349,  47.   ],
       [  1.   ,  93.   ,  70.   , ...,  30.4  ,   0.315,  23.   ]])

In [15]:
y = df['class'].values
y

array([1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0,
       1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1,
       0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0,
       1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1,
       0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1,
       1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1,
       1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0,
       1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0,
       1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0,
       0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1,

In [16]:
estimator = DecisionTreeClassifier(max_depth=3,random_state=42)

bag_clf = BaggingClassifier(
    estimator = estimator,
    n_estimators=100, # Cantidad de árboles
    max_samples=50, # Muestras utilizadas en bootstrapping
    bootstrap=True, # Usamos bootstrapping: muestreo con reemplazo.
    max_features = 4, # Features que utiliza en el bootstrapping. Cuanto más bajo, mejor generalizará y menos overfitting
    random_state=42)

bag_clf.fit(X, y)
y_pred = bag_clf.predict(X)
accuracy_score(y, y_pred)
print("Accuracy:", accuracy_score(y, y_pred))
print("\nClassification Report:\n", classification_report(y, y_pred))

Accuracy: 0.7747395833333334

Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.92      0.84       500
           1       0.78      0.50      0.61       268

    accuracy                           0.77       768
   macro avg       0.78      0.71      0.72       768
weighted avg       0.78      0.77      0.76       768



In [17]:
kfold = KFold(
            n_splits=10,
            shuffle=True,
            random_state=42
        )

scores = cross_val_score(
            bag_clf,
            X,
            y,
            cv=kfold,
            scoring="accuracy"
        )

print(f"Accuracy scores for each fold: {scores}")
print(f"Mean accuracy: {scores.mean():.4f}")
print(f"Standard deviation: {scores.std():.4f}")

Accuracy scores for each fold: [0.71428571 0.79220779 0.7012987  0.80519481 0.81818182 0.68831169
 0.80519481 0.77922078 0.65789474 0.77631579]
Mean accuracy: 0.7538
Standard deviation: 0.0546


### 4. Random Forest
En este caso entrena un [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) con 100 árboles y un `max_features` de 3. También con validación cruzada

In [18]:
from sklearn.ensemble import RandomForestClassifier

In [19]:
from sklearn.model_selection import StratifiedKFold

In [20]:
rf_clf = RandomForestClassifier(n_estimators=100,
                                 max_features=3,
                                 random_state=42)

# StratifiedKFold ensures balanced class distribution in each fold
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Perform cross-validation
scores_rf = cross_val_score(rf_clf, X, y, cv=cv, scoring='accuracy')

print(f"Accuracy scores for each fold: {scores_rf}")
print(f"Mean accuracy: {scores_rf.mean():.4f}")
print(f"Standard deviation: {scores_rf.std():.4f}")
print()

# Train final model on all data
rf_clf.fit(X, y)

y_pred_rf = rf_clf.predict(X)
# np.sum(y_test == y_pred_rf) / len(y_test) 
print("Accuracy:", accuracy_score(y, y_pred_rf))
print("\nClassification Report:\n", classification_report(y, y_pred_rf))

Accuracy scores for each fold: [0.77922078 0.74025974 0.72727273 0.83116883 0.80519481 0.75324675
 0.75324675 0.72727273 0.76315789 0.65789474]
Mean accuracy: 0.7538
Standard deviation: 0.0449

Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       500
           1       1.00      1.00      1.00       268

    accuracy                           1.00       768
   macro avg       1.00      1.00      1.00       768
weighted avg       1.00      1.00      1.00       768



### 5. AdaBoost
Implementa un [AdaBoostClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html) con 30 árboles.

In [21]:
from sklearn.ensemble import AdaBoostClassifier

In [22]:
base_tree = DecisionTreeClassifier(max_depth=1, random_state=42)

ada_clf = AdaBoostClassifier(estimator = base_tree,
                             n_estimators=30,
                             learning_rate=0.8,
                             random_state=42)

# Perform 10-fold cross-validation
cv_scores_ada = cross_val_score(
    ada_clf, X, y,
    cv=10, scoring='accuracy'
)

print(f"Accuracy scores for each fold: {cv_scores_ada}")
print(f"Mean accuracy: {cv_scores_ada.mean():.4f}")
print(f"Standard deviation: {cv_scores_ada.std():.4f}")
print()

ada_clf.fit(X, y)

y_pred_ada = ada_clf.predict(X)
print("Accuracy:", accuracy_score(y, y_pred_ada))
print("\nClassification Report:\n", classification_report(y, y_pred_ada))

Accuracy scores for each fold: [0.71428571 0.77922078 0.75324675 0.64935065 0.74025974 0.74025974
 0.80519481 0.77922078 0.75       0.84210526]
Mean accuracy: 0.7553
Standard deviation: 0.0496

Accuracy: 0.8072916666666666

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.89      0.86       500
           1       0.76      0.66      0.71       268

    accuracy                           0.81       768
   macro avg       0.79      0.77      0.78       768
weighted avg       0.80      0.81      0.80       768



### 6. GradientBoosting
Implementa un [GradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html) con 100 estimadores

In [23]:
from sklearn.ensemble import GradientBoostingClassifier

In [24]:
gb_clf = GradientBoostingClassifier(
    n_estimators=100,     # number of boosting stages (trees)
    learning_rate=0.1,   # step size shrinkage
    max_depth=2,         # depth of individual trees
    random_state=42
)

# Perform 10-fold cross-validation
cv_scores_gb = cross_val_score(
    gb_clf, X, y,
    cv=10, scoring='accuracy'
)

print(f"Accuracy scores for each fold: {cv_scores_gb}")
print(f"Mean accuracy: {cv_scores_gb.mean():.4f}")
print(f"Standard deviation: {cv_scores_gb.std():.4f}")
print()

gb_clf.fit(X, y)

y_pred_gb = gb_clf.predict(X)
print("Accuracy:", accuracy_score(y, y_pred_gb))
print("\nClassification Report:\n", classification_report(y, y_pred_gb))

Accuracy scores for each fold: [0.74025974 0.79220779 0.80519481 0.7012987  0.75324675 0.79220779
 0.80519481 0.80519481 0.71052632 0.78947368]
Mean accuracy: 0.7695
Standard deviation: 0.0381

Accuracy: 0.8385416666666666

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.92      0.88       500
           1       0.82      0.69      0.75       268

    accuracy                           0.84       768
   macro avg       0.83      0.80      0.81       768
weighted avg       0.84      0.84      0.83       768



### 7. XGBoost
Para este apartado utiliza un [XGBoostClassifier](https://docs.getml.com/latest/api/getml.predictors.XGBoostClassifier.html) con 100 estimadores. XGBoost no forma parte de la suite de modelos de sklearn, por lo que tendrás que instalarlo con pip install

In [25]:
from xgboost import XGBClassifier

In [26]:
xgb_clf = XGBClassifier(
    n_estimators=100,      # number of boosting stages (trees)
    booster='gbtree',
    learning_rate=0.1,     # step size shrinkage
    max_depth=3,           # depth of individual trees
    random_state=42
)

# Perform 10-fold cross-validation
cv_scores_xgb = cross_val_score(
    xgb_clf, X, y,
    cv=10, scoring='accuracy'
)

print(f"Accuracy scores for each fold: {cv_scores_xgb}")
print(f"Mean accuracy: {cv_scores_xgb.mean():.4f}")
print(f"Standard deviation: {cv_scores_xgb.std():.4f}")
print()

xgb_clf.fit(X, y)

y_pred_xgb = xgb_clf.predict(X)
print("Accuracy:", accuracy_score(y, y_pred_xgb))
print("\nClassification Report:\n", classification_report(y, y_pred_xgb))

Accuracy scores for each fold: [0.76623377 0.80519481 0.71428571 0.63636364 0.72727273 0.79220779
 0.81818182 0.84415584 0.67105263 0.81578947]
Mean accuracy: 0.7591
Standard deviation: 0.0656

Accuracy: 0.875

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.93      0.91       500
           1       0.85      0.78      0.81       268

    accuracy                           0.88       768
   macro avg       0.87      0.85      0.86       768
weighted avg       0.87      0.88      0.87       768



### 8. Primeros resultados
Crea un dataframe con los resultados y sus algoritmos, ordenándolos de mayor a menor

In [33]:
rsts = [['Bagging', bag_clf, scores.mean(), accuracy_score(y, y_pred), precision_score(y, y_pred), recall_score(y, y_pred), f1_score(y, y_pred)], 
        ['RandomForest', rf_clf, scores_rf.mean(), accuracy_score(y, y_pred_rf), precision_score(y, y_pred_rf), recall_score(y, y_pred_rf), f1_score(y, y_pred_rf)], 
        ['AdaBoost', ada_clf, cv_scores_ada.mean(), accuracy_score(y, y_pred_ada), precision_score(y, y_pred_ada), recall_score(y, y_pred_ada), f1_score(y, y_pred_ada)], 
        ['GradBoost', gb_clf, cv_scores_gb.mean(), accuracy_score(y, y_pred_gb), precision_score(y, y_pred_gb), recall_score(y, y_pred_gb), f1_score(y, y_pred_gb)], 
        ['XGBoost', xgb_clf, cv_scores_xgb.mean(), accuracy_score(y, y_pred_xgb), precision_score(y, y_pred_xgb), recall_score(y, y_pred_xgb), f1_score(y, y_pred_xgb)]
]

df_scores = pd.DataFrame(data = rsts, columns=['model_descr', 'model_obj', 'cv_accuracy_mean', 'accuracy', 'precision', 'recall', 'f1_score'])
df_scores.sort_values(by='accuracy', ascending=False)

,model_descr,model_obj,cv_accuracy_mean,accuracy,precision,recall,f1_score
1,RandomForest,"(DecisionTreeClassifier(max_features=3, random...",0.753794,1.000000,1.000000,1.000000,1.000000
4,XGBoost,"XGBClassifier(base_score=None, booster='gbtree...",0.759074,0.875000,0.849593,0.779851,0.813230
3,GradBoost,([DecisionTreeRegressor(criterion='friedman_ms...,0.769481,0.838542,0.821429,0.686567,0.747967
2,AdaBoost,"(DecisionTreeClassifier(max_depth=1, random_st...",0.755314,0.807292,0.756410,0.660448,0.705179
0,Bagging,"(DecisionTreeClassifier(max_depth=3, random_st...",0.753811,0.774740,0.777778,0.496269,0.605923


### 9. Hiperparametrización
Vuelve a entrenar los modelos de nuevo, pero esta vez dividiendo el conjunto de datos en train/test y utilizando un gridsearch para encontrar los mejores hiperparámetros.

In [36]:
from sklearn.model_selection import train_test_split, GridSearchCV

In [139]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state=42)

Bagging

In [140]:
param_grid_bag = {
    'n_estimators': [100, 200, 400],
    'max_samples': [0.2, 0.5, 0.8],
    'max_features': [0.2, 0.5, 0.8]
}

In [141]:
grid_search_bag = GridSearchCV(
    estimator=bag_clf,
    param_grid=param_grid_bag,
    scoring='accuracy',  # Optimize for accuracy
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search_bag.fit(X_train, y_train)
print("Mejores parámetros Bagging:", grid_search_bag.best_params_)
bag_clf_best = grid_search_bag.best_estimator_

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Mejores parámetros Bagging: {'max_features': 0.8, 'max_samples': 0.8, 'n_estimators': 400}


In [142]:
y_pred_train_bag = bag_clf_best.predict(X_train)
y_pred_test_bag = bag_clf_best.predict(X_test)

acc_bag = accuracy_score(y_test, y_pred_test_bag)
prec_bag = precision_score(y_test, y_pred_test_bag)
rec_bag = recall_score(y_test, y_pred_test_bag)
f1_bag = f1_score(y_test, y_pred_test_bag)

RandomForest

In [143]:
param_grid_rf = {
    'n_estimators': [100, 150, 200],
    'max_depth': [2, 3, 4, 5],
    'max_features': [3, 5, 7],
    'min_samples_split': [2, 3, 4],
    'min_samples_leaf': [1, 2, 3]
}

In [144]:
grid_search_rf = GridSearchCV(
    estimator=rf_clf,
    param_grid=param_grid_rf,
    scoring='accuracy',  # Optimize for accuracy
    cv=5,
    n_jobs=-1,
    verbose=3
)

grid_search_rf.fit(X_train, y_train)
print("Mejores parámetros RandomForest:", grid_search_rf.best_params_)
rf_clf_best = grid_search_rf.best_estimator_

Fitting 5 folds for each of 324 candidates, totalling 1620 fits
Mejores parámetros RandomForest: {'max_depth': 5, 'max_features': 5, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}


In [145]:
y_pred_train_rf = rf_clf_best.predict(X_train)
y_pred_test_rf = rf_clf_best.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_test_rf)
prec_rf = precision_score(y_test, y_pred_test_rf)
rec_rf = recall_score(y_test, y_pred_test_rf)
f1_rf = f1_score(y_test, y_pred_test_rf)

AdaBoost

In [146]:
base_tree1 = DecisionTreeClassifier(max_depth=1, random_state=42)
base_tree2 = DecisionTreeClassifier(max_depth=2, random_state=42)
# base_tree3 = DecisionTreeClassifier(max_depth=3, random_state=42)

param_grid_ab = {
    'estimator': [base_tree1, base_tree2],
    'n_estimators': [30, 60, 100],
    'learning_rate': [0.01, 0.1, 0.2, 0.5, 0.8, 1., 1.2],
    'algorithm': ['SAMME', 'SAMME.R']
}

In [147]:
grid_search_ab = GridSearchCV(
    estimator=AdaBoostClassifier(random_state=42),
    param_grid=param_grid_ab,
    scoring='accuracy',  # Optimize for accuracy
    cv=5,
    n_jobs=-1,
    verbose=3
)

grid_search_ab.fit(X_train, y_train)
print("Mejores parámetros AdaBoost:", grid_search_ab.best_params_)
ab_clf_best = grid_search_ab.best_estimator_

Fitting 5 folds for each of 84 candidates, totalling 420 fits
Mejores parámetros AdaBoost: {'algorithm': 'SAMME', 'estimator': DecisionTreeClassifier(max_depth=1, random_state=42), 'learning_rate': 1.0, 'n_estimators': 100}


In [148]:
y_pred_train_ab = ab_clf_best.predict(X_train)
y_pred_test_ab = ab_clf_best.predict(X_test)

acc_ab = accuracy_score(y_test, y_pred_test_ab)
prec_ab = precision_score(y_test, y_pred_test_ab)
rec_ab = recall_score(y_test, y_pred_test_ab)
f1_ab = f1_score(y_test, y_pred_test_ab)

GradientBoost

In [149]:
param_grid_gb = {
    'n_estimators': [20, 50, 100],
    'learning_rate': [0.05, 0.1, 0.2, 0.5, 1.],
    'max_depth': [1, 2]
}

In [150]:
grid_search_gb = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid=param_grid_gb,
    scoring='accuracy',  # Optimize for accuracy
    cv=5,
    n_jobs=-1,
    verbose=3
)

grid_search_gb.fit(X_train, y_train)
print("Mejores parámetros GradientBoost:", grid_search_gb.best_params_)
gb_clf_best = grid_search_gb.best_estimator_

Fitting 5 folds for each of 30 candidates, totalling 150 fits


Mejores parámetros GradientBoost: {'learning_rate': 0.2, 'max_depth': 1, 'n_estimators': 20}


In [151]:
y_pred_train_gb = gb_clf_best.predict(X_train)
y_pred_test_gb = gb_clf_best.predict(X_test)

acc_gb = accuracy_score(y_test, y_pred_test_gb)
prec_gb = precision_score(y_test, y_pred_test_gb)
rec_gb = recall_score(y_test, y_pred_test_gb)
f1_gb = f1_score(y_test, y_pred_test_gb)

XGBoost

In [152]:
param_grid_xgb = {
    'n_estimators': [10, 20, 50, 100],
    'learning_rate': [0.05, 0.1, 0.2, 0.5, 1., 1.2, 1.5],
    'max_depth': [1, 2, 3, 4]
}

In [166]:
grid_search_xgb = GridSearchCV(
    estimator=XGBClassifier(booster='gbtree', random_state=42),
    param_grid=param_grid_xgb,
    scoring='accuracy',  # Optimize for accuracy
    cv=5,
    n_jobs=-1,
    verbose=3
)

grid_search_xgb.fit(X_train, y_train)
print("Mejores parámetros XGBoost:", grid_search_xgb.best_params_)
xgb_clf_best = grid_search_xgb.best_estimator_

Fitting 5 folds for each of 112 candidates, totalling 560 fits
Mejores parámetros XGBoost: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}


In [167]:
y_pred_train_xgb = xgb_clf_best.predict(X_train)
y_pred_test_xgb = xgb_clf_best.predict(X_test)

acc_xgb = accuracy_score(y_test, y_pred_test_xgb)
prec_xgb = precision_score(y_test, y_pred_test_xgb)
rec_xgb = recall_score(y_test, y_pred_test_xgb)
f1_xgb = f1_score(y_test, y_pred_test_xgb)

Resumen de resultados tras optimización de modelos con grid search

In [168]:
rsts_opt = [['Bagging', bag_clf_best, grid_search_bag.best_score_, acc_bag, prec_bag, rec_bag, f1_bag], 
        ['RandomForest', rf_clf_best, grid_search_rf.best_score_, acc_rf, prec_rf, rec_rf, f1_rf], 
        ['AdaBoost', ab_clf_best, grid_search_ab.best_score_, acc_ab, prec_ab, rec_ab, f1_ab], 
        ['GradBoost', gb_clf_best, grid_search_gb.best_score_, acc_gb, prec_gb, rec_gb, f1_gb], 
        ['XGBoost', xgb_clf_best, grid_search_xgb.best_score_, acc_xgb, prec_xgb, rec_xgb, f1_xgb]
]

df_scores_opt = pd.DataFrame(data = rsts_opt, columns=['model_descr', 'model_obj', 'cv_accuracy_mean', 'accuracy', 'precision', 'recall', 'f1_score'])
df_scores_opt.sort_values(by='accuracy', ascending=False)

,model_descr,model_obj,cv_accuracy_mean,accuracy,precision,recall,f1_score
0,Bagging,"(DecisionTreeClassifier(max_depth=3, random_st...",0.770385,0.779221,0.723404,0.618182,0.666667
3,GradBoost,([DecisionTreeRegressor(criterion='friedman_ms...,0.780168,0.772727,0.738095,0.563636,0.639175
1,RandomForest,"(DecisionTreeClassifier(max_depth=5, max_featu...",0.788285,0.759740,0.660714,0.672727,0.666667
4,XGBoost,"XGBClassifier(base_score=None, booster='gbtree...",0.783420,0.753247,0.654545,0.654545,0.654545
2,AdaBoost,"(DecisionTreeClassifier(max_depth=1, random_st...",0.780155,0.740260,0.627119,0.672727,0.649123


In [169]:
y_pred_xgb_2 = xgb_clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_xgb_2))

Accuracy: 0.8701298701298701


### 10. Conclusiones finales

El modelo con el que se ha conseguido mejor accuracy en los datos de test en este caso ha sido el clasificador bagging con árbol de decisión como estimador. 